# Summary_Day12_offline.ipynb  
## 튜닝 · 최적화 · 과적합 대응 · 인터넷 불가 버전 · 합성 CIFAR-10식 데이터

이 파일은 **인터넷이 안 되는 환경**에서 12강 튜닝 흐름을 실행하기 위한 버전이다.

원본 강의는 CIFAR-10을 다운로드해서 사용한다.  
하지만 인터넷이 없으면 `datasets.CIFAR10(download=True)`가 실패할 수 있다.

그래서 이 파일은 다운로드 없이 실행되도록 **합성 컬러 이미지 데이터**를 만든다.

```text
원본 강의: CIFAR-10, [3, 32, 32], 10 class
인터넷 불가 버전: synthetic CIFAR-like, [3, 32, 32], 10 class
```

데이터는 다르지만 코드 흐름은 같다.

```text
CNN_v2 깊은 모델
→ Optimizer 비교
→ Dropout 추가
→ BatchNorm 추가
→ Data Augmentation 흉내
→ train/eval 차이
→ Softmax 확률 분석
```

> 필기 포인트:  
> 인터넷이 없을 때도 튜닝 개념은 끊기면 안 된다.  
> 데이터셋만 합성으로 바꾸고, 모델/학습/평가 구조는 강의와 같은 방식으로 연습한다.

## 1. 라이브러리 준비

다운로드 데이터셋을 쓰지 않으므로 `torchvision.datasets`는 필요 없다.  
합성 이미지는 NumPy로 만들고, PyTorch의 `TensorDataset`으로 묶는다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix


torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

n_output = len(classes)

print("device:", device)
print("class 수:", n_output)

## 2. 합성 CIFAR-10식 데이터 만들기

CIFAR-10처럼 3채널 32×32 이미지를 만든다.

각 class마다 색상 channel, 선 위치, 대각선 방향을 다르게 주어 분류 가능한 패턴을 만든다.

```text
입력 shape: [N, 3, 32, 32]
label: 0~9
```

In [ ]:
def make_synthetic_cifar_like(n_per_class=120, noise_level=0.12):
    images = []
    labels = []

    for cls in range(10):
        for _ in range(n_per_class):
            img = np.random.normal(0.0, noise_level, size=(3, 32, 32)).astype(np.float32)

            channel = cls % 3
            row = 4 + (cls * 2) % 20
            col = 4 + (cls * 3) % 20

            img[channel, row:row + 6, :] += 1.0
            img[channel, :, col:col + 4] += 0.7

            if cls % 2 == 0:
                for i in range(8, 24):
                    img[channel, i, i] += 0.8
            else:
                for i in range(8, 24):
                    img[channel, i, 31 - i] += 0.8

            img = np.clip(img, -1.0, 1.0)

            images.append(img)
            labels.append(cls)

    X = np.stack(images)
    y = np.array(labels)

    indices = np.random.permutation(len(y))

    return X[indices], y[indices]

X, y = make_synthetic_cifar_like(n_per_class=120)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("class 분포:", np.bincount(y))

## 3. 이미지 시각화 함수

PyTorch 이미지는 CHW 순서다.  
Matplotlib은 HWC 순서를 기대하므로 `permute` 또는 `transpose`로 순서를 바꿔야 한다.

In [ ]:
def show_tensor_images(X_tensor, y_tensor, pred_tensor=None, n_show=20):
    plt.figure(figsize=(10, 4))

    for i in range(n_show):
        plt.subplot(2, 10, i + 1)

        img = X_tensor[i].permute(1, 2, 0).numpy()
        img = (img + 1.0) / 2.0
        img = np.clip(img, 0, 1)

        true_label = int(y_tensor[i])

        if pred_tensor is None:
            title = classes[true_label]
            color = "black"
        else:
            pred_label = int(pred_tensor[i])
            title = f"{classes[true_label]}\n→ {classes[pred_label]}"
            color = "black" if true_label == pred_label else "red"

        plt.imshow(img)
        plt.title(title, fontsize=8, color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

X_tensor_all = torch.FloatTensor(X)
y_tensor_all = torch.LongTensor(y)

show_tensor_images(X_tensor_all, y_tensor_all, n_show=20)

## 4. train/test 분할과 DataLoader

합성 데이터를 80% train, 20% test로 나눈다.

In [ ]:
n_total = len(y)
n_train = int(n_total * 0.8)

X_train = torch.FloatTensor(X[:n_train])
X_test = torch.FloatTensor(X[n_train:])

y_train = torch.LongTensor(y[:n_train])
y_test = torch.LongTensor(y[n_train:])

batch_size = 64

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("train batch:", len(train_loader))
print("test batch:", len(test_loader))

## 5. Optimizer 기본 개념 확인

SGD 수동 업데이트와 PyTorch Optimizer 생성을 다시 확인한다.

In [ ]:
W = torch.randn(3, 3, requires_grad=True)
B = torch.randn(3, requires_grad=True)

W.grad = torch.randn(3, 3)
B.grad = torch.randn(3)

lr = 0.001

W.data -= lr * W.grad.data
B.data -= lr * B.grad.data

net_params = [torch.randn(10, 1, requires_grad=True)]

optimizer_momentum = optim.SGD(net_params, lr=lr, momentum=0.9)
optimizer_adam = optim.Adam(net_params)

print("Momentum Optimizer:")
print(optimizer_momentum)

print("\nAdam Optimizer:")
print(optimizer_adam)

## 6. Dropout train/eval 동작 확인

Dropout은 학습 모드와 평가 모드에서 다르게 동작한다.

In [ ]:
torch.manual_seed(123)

inputs = torch.randn(1, 10)
dropout = nn.Dropout(0.5)

dropout.train()
outputs_train = dropout(inputs)

dropout.eval()
outputs_eval = dropout(inputs)

print("Original:")
print(inputs)

print("\nTrain Mode:")
print(outputs_train)

print("\nEval Mode:")
print(outputs_eval)

## 7. 합성 데이터 증강 함수

인터넷 불가 버전에서는 `transforms` 대신 Tensor에 직접 간단한 증강을 적용한다.

적용할 증강은 다음이다.

```text
좌우 반전
랜덤 영역 지우기
```

In [ ]:
def augment_batch(images, flip_prob=0.5, erase_prob=0.5):
    images = images.clone()

    batch_size = images.size(0)

    for i in range(batch_size):
        if np.random.rand() < flip_prob:
            images[i] = torch.flip(images[i], dims=[2])

        if np.random.rand() < erase_prob:
            h = np.random.randint(4, 12)
            w = np.random.randint(4, 12)
            top = np.random.randint(0, 32 - h)
            left = np.random.randint(0, 32 - w)
            images[i, :, top:top + h, left:left + w] = 0.0

    return images

aug_sample = augment_batch(X_train[:20], flip_prob=0.8, erase_prob=0.8)

show_tensor_images(aug_sample, y_train[:20], n_show=20)

## 8. 공통 학습 함수 만들기

증강을 적용할지 선택할 수 있는 학습 함수를 만든다.

In [ ]:
def torch_seed(seed=123):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def fit(net, optimizer, criterion, num_epochs, train_loader, test_loader, device, use_augmentation=False):
    history = []

    for epoch in range(num_epochs):
        net.train()

        train_loss = 0.0
        train_acc = 0.0
        n_train = 0

        for inputs, labels in train_loader:
            if use_augmentation:
                inputs = augment_batch(inputs)

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            predicted = torch.max(outputs, 1)[1]

            train_loss += loss.item() * labels.size(0)
            train_acc += (predicted == labels).sum().item()
            n_train += labels.size(0)

        net.eval()

        val_loss = 0.0
        val_acc = 0.0
        n_val = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                predicted = torch.max(outputs, 1)[1]

                val_loss += loss.item() * labels.size(0)
                val_acc += (predicted == labels).sum().item()
                n_val += labels.size(0)

        history.append([
            epoch + 1,
            train_loss / n_train,
            train_acc / n_train,
            val_loss / n_val,
            val_acc / n_val
        ])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={history[-1][1]:.4f} | train_acc={history[-1][2]:.4f} | "
            f"val_loss={history[-1][3]:.4f} | val_acc={history[-1][4]:.4f}"
        )

    return np.array(history)

In [ ]:
def evaluate_history(history, title="Learning Curve"):
    plt.plot(history[:, 0], history[:, 1], label="train loss")
    plt.plot(history[:, 0], history[:, 3], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title + " Loss")
    plt.legend()
    plt.show()

    plt.plot(history[:, 0], history[:, 2], label="train acc")
    plt.plot(history[:, 0], history[:, 4], label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

    print("최종 검증 정확도:", history[-1, 4])


def collect_predictions(net, loader, device):
    net.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = net(inputs)
            predicted = torch.max(outputs, 1)[1].cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(predicted)

    return np.array(y_true), np.array(y_pred)

## 9. CNN_v2 깊은 모델

강의와 같은 구조의 깊은 CNN을 만든다.

```text
conv1-2: 32채널
conv3-4: 64채널
conv5-6: 128채널
pooling 3번
```

In [ ]:
class CNN_v2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.l1 = nn.Linear(4 * 4 * 128, 128)
        self.l2 = nn.Linear(128, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.relu,
            self.conv2, self.relu,
            self.maxpool,

            self.conv3, self.relu,
            self.conv4, self.relu,
            self.maxpool,

            self.conv5, self.relu,
            self.conv6, self.relu,
            self.maxpool
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

net_v2 = CNN_v2(n_output).to(device)

dummy = torch.randn(2, 3, 32, 32).to(device)

with torch.no_grad():
    feature = net_v2.features(dummy)
    output = net_v2(dummy)

print("feature shape:", feature.shape)
print("output shape:", output.shape)

## 10. Optimizer 비교: SGD / Momentum / Adam

같은 모델에 Optimizer만 바꿔서 비교한다.

In [ ]:
num_epochs = 3

histories = {}

for opt_name in ["SGD", "Momentum", "Adam"]:
    torch_seed()

    net = CNN_v2(n_output).to(device)
    criterion = nn.CrossEntropyLoss()

    if opt_name == "SGD":
        optimizer = optim.SGD(net.parameters(), lr=0.03)
    elif opt_name == "Momentum":
        optimizer = optim.SGD(net.parameters(), lr=0.03, momentum=0.9)
    else:
        optimizer = optim.Adam(net.parameters(), lr=0.001)

    print("\n===", opt_name, "===")

    history = fit(
        net,
        optimizer,
        criterion,
        num_epochs,
        train_loader,
        test_loader,
        device
    )

    histories[opt_name] = history

In [ ]:
for name, history in histories.items():
    plt.plot(history[:, 0], history[:, 4], label=name)

plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Optimizer Comparison")
plt.legend()
plt.show()

## 11. CNN_v3: Dropout 추가

Dropout을 pool 뒤와 classifier 안에 넣는다.

In [ ]:
class CNN_v3(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.4)

        self.l1 = nn.Linear(4 * 4 * 128, 128)
        self.l2 = nn.Linear(128, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.relu,
            self.conv2, self.relu, self.maxpool,
            self.dropout1,

            self.conv3, self.relu,
            self.conv4, self.relu, self.maxpool,
            self.dropout2,

            self.conv5, self.relu,
            self.conv6, self.relu, self.maxpool,
            self.dropout3
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.dropout3,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

## 12. CNN_v4: BatchNorm 추가

BatchNorm은 `Conv → BatchNorm → ReLU` 순서로 넣는다.

채널 수가 같아도 각 위치마다 별도 BatchNorm 인스턴스를 만든다.

In [ ]:
class CNN_v4(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)
        self.bn5 = nn.BatchNorm2d(128)
        self.bn6 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((2, 2))
        self.flatten = nn.Flatten()

        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.4)

        self.l1 = nn.Linear(4 * 4 * 128, 512)
        self.l2 = nn.Linear(512, num_classes)

        self.features = nn.Sequential(
            self.conv1, self.bn1, self.relu,
            self.conv2, self.bn2, self.relu, self.maxpool,
            self.dropout1,

            self.conv3, self.bn3, self.relu,
            self.conv4, self.bn4, self.relu, self.maxpool,
            self.dropout2,

            self.conv5, self.bn5, self.relu,
            self.conv6, self.bn6, self.relu, self.maxpool,
            self.dropout3
        )

        self.classifier = nn.Sequential(
            self.l1,
            self.relu,
            self.dropout3,
            self.l2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

## 13. CNN_v4 + 증강 학습

최종 조합은 다음이다.

```text
CNN_v4
+ Adam
+ Dropout
+ BatchNorm
+ Data Augmentation
```

In [ ]:
torch_seed()

net_final = CNN_v4(n_output).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net_final.parameters(), lr=0.001)

history_final = fit(
    net_final,
    optimizer,
    criterion,
    num_epochs=5,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    use_augmentation=True
)

evaluate_history(history_final, title="CNN_v4 + Synthetic Augmentation")

## 14. 최종 예측 이미지 확인

정답과 예측을 함께 확인한다.

In [ ]:
y_true, y_pred = collect_predictions(net_final, test_loader, device)

show_tensor_images(X_test, y_test, torch.LongTensor(y_pred), n_show=20)

## 15. Softmax 확률 분석

개별 이미지 하나에 대해 class별 확률을 출력한다.

In [ ]:
net_final.eval()

sample_image = X_test[0]
sample_label = y_test[0]

with torch.no_grad():
    output = net_final(sample_image.view(1, 3, 32, 32).to(device))
    probs = torch.softmax(output, dim=1)

probs_np = probs.cpu().numpy()[0]

names = np.array(classes)
values = np.array([f"{x:.4f}" for x in probs_np])

tbl = np.array([names, values]).T

print("true:", classes[int(sample_label)])
print("pred:", classes[int(np.argmax(probs_np))])
print(tbl)

## 16. BatchNorm 내부 동작 확인

BatchNorm은 train mode와 eval mode에서 다른 통계를 사용한다.

In [ ]:
torch.manual_seed(123)

bn_inputs = torch.randn(1, 1, 10)

i_mean = bn_inputs.mean()
i_var = bn_inputs.var(unbiased=True)
i_std = bn_inputs.std(unbiased=False)

bn = nn.BatchNorm1d(1)

bn.train()
outputs_train = bn(bn_inputs)

bn.eval()
outputs_eval = bn(bn_inputs)

print("입력 평균:", round(i_mean.item(), 4))
print("입력 표준편차:", round(i_std.item(), 4))

print("\n훈련 출력:")
print(outputs_train.data.numpy().round(4))

print("\n평가 출력:")
print(outputs_eval.data.numpy().round(4))

print("\nrunning_mean:", bn.running_mean.data.numpy().round(4))
print("running_var:", bn.running_var.data.numpy().round(4))

## 17. 최종 평가 리포트

합성 데이터 기준 최종 성능을 확인한다.

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Synthetic CIFAR-like Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(10), classes, rotation=45, ha="right")
plt.yticks(range(10), classes)

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=7)

plt.colorbar()
plt.tight_layout()
plt.show()

## 18. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `tuning` | 성능 개선 설정 조정 | optimizer, dropout 등 |
| `overfitting` | 훈련 데이터에 과하게 맞는 현상 | val 성능 하락 |
| `SGD` | 기본 Optimizer | `optim.SGD(...)` |
| `Momentum` | 이전 방향 반영 | `momentum=0.9` |
| `Adam` | 안정적 Optimizer | `optim.Adam(...)` |
| `Dropout` | 뉴런 일부 비활성화 | `nn.Dropout(p)` |
| `BatchNorm2d` | feature map 정규화 | `nn.BatchNorm2d(ch)` |
| `Data Augmentation` | 데이터 변형 | flip, erasing |
| `train()` | 학습 모드 | Dropout 적용, BN 갱신 |
| `eval()` | 평가 모드 | Dropout 중단, BN running 통계 사용 |
| `softmax` | logits를 확률로 변환 | `torch.softmax(output, dim=1)` |
| `TensorDataset` | Tensor 기반 Dataset | 인터넷 없이 데이터 구성 |
| `DataLoader` | mini-batch 공급 | `DataLoader(...)` |

## 19. 시험용 요약

```text
12강 핵심 = 깊은 CNN의 성능을 높이기 위한 튜닝과 과적합 대응
```

꼭 기억할 것:

- 깊은 모델은 복잡한 특징을 학습할 수 있다.
- 깊다고 무조건 성능이 좋아지는 것은 아니다.
- Optimizer 선택은 수렴 속도와 최종 성능에 영향을 준다.
- SGD는 기본 Optimizer다.
- Momentum은 이전 이동 방향을 반영한다.
- Adam은 안정적이고 빠른 수렴을 기대할 수 있다.
- 과적합은 train 성능은 좋아지는데 validation 성능이 나빠지는 현상이다.
- Dropout은 일부 뉴런을 꺼서 과도한 의존을 막는다.
- Dropout은 train mode에서만 적용된다.
- BatchNorm은 feature map 분포를 안정화한다.
- BatchNorm은 Conv 뒤, ReLU 앞에 두는 것이 표준적이다.
- BatchNorm 인스턴스는 재사용하지 말고 위치마다 따로 만든다.
- Data Augmentation은 데이터를 변형해 일반화 성능을 높인다.
- `train()`과 `eval()`은 Dropout, BatchNorm 동작을 바꾸므로 반드시 구분한다.
- Softmax 확률은 모델이 어떤 class와 헷갈렸는지 분석할 때 사용한다.